In [1]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 129)


In [3]:
def growth_label(score):

    if score < 35:
        return "Low"

    elif score < 60:
        return "Medium"

    else:
        return "High"


df["growth_potential_label"] = (
    df["growth_score"]
    .apply(growth_label)
)

print(
    df["growth_potential_label"]
    .value_counts()
)

growth_potential_label
Medium    27011
High      16032
Low        6957
Name: count, dtype: int64


In [4]:
features = [

    "employee_growth_rate",

    "customer_growth_rate",

    "revenue_growth_rate",

    "monthly_active_users",

    "retention_rate",

    "customer_count",

    "revenue",

    "employee_count"

]

X = df[features]

y = df["growth_potential_label"]

In [5]:
print(X.dtypes)

print(
    X.isnull()
    .sum()
)

employee_growth_rate    float64
customer_growth_rate    float64
revenue_growth_rate     float64
monthly_active_users      int64
retention_rate          float64
customer_count            int64
revenue                   int64
employee_count            int64
dtype: object
employee_growth_rate    0
customer_growth_rate    0
revenue_growth_rate     0
monthly_active_users    0
retention_rate          0
customer_count          0
revenue                 0
employee_count          0
dtype: int64


In [6]:
le = LabelEncoder()

y = le.fit_transform(y)

print(
    dict(
        zip(
            le.classes_,
            range(
                len(le.classes_)
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


In [7]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [8]:
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=12, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [9]:
preds = model.predict(X_test)

acc = accuracy_score(
    y_test,
    preds
)

print(
    "Accuracy:",
    round(acc*100,2),
    "%"
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 94.58 %
              precision    recall  f1-score   support

           0       0.98      0.93      0.95      3207
           1       0.99      0.83      0.90      1391
           2       0.92      0.98      0.95      5402

    accuracy                           0.95     10000
   macro avg       0.96      0.91      0.94     10000
weighted avg       0.95      0.95      0.95     10000



In [10]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
        model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(importance)

                Feature  Importance
1  customer_growth_rate    0.298828
2   revenue_growth_rate    0.291522
0  employee_growth_rate    0.206219
4        retention_rate    0.130018
5        customer_count    0.019571
3  monthly_active_users    0.019196
6               revenue    0.018327
7        employee_count    0.016319


In [12]:
joblib.dump(

    model,

    "../models/growth_potential_model/growth_potential_model.pkl"

)

joblib.dump(

    le,

    "../models/growth_potential_model/label_encoder.pkl"

)

print("Growth Potential Model Saved ✅")

Growth Potential Model Saved ✅


In [13]:
metadata = {

    "model_name":
        "Growth Potential Model",

    "algorithm":
        "Random Forest",

    "features":
        features,

    "classes":
        list(
            le.classes_
        )

}

with open(

    "../models/growth_potential_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [14]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("Dataset Saved ✅")

Dataset Saved ✅


In [15]:
print(df["growth_potential_label"].value_counts())

print("Accuracy:", acc)

print(importance)

growth_potential_label
Medium    27011
High      16032
Low        6957
Name: count, dtype: int64
Accuracy: 0.9458
                Feature  Importance
1  customer_growth_rate    0.298828
2   revenue_growth_rate    0.291522
0  employee_growth_rate    0.206219
4        retention_rate    0.130018
5        customer_count    0.019571
3  monthly_active_users    0.019196
6               revenue    0.018327
7        employee_count    0.016319
